# Weld-YOLO11-SO: Dual Environment (Local & Google Colab) Training Notebook

This notebook is fully configured for **seamless execution BOTH locally** (inside `main_tranning/` subfolder) **AND on Google Colab** (with GPU acceleration).

### Weld-YOLO11-SO Highlights:
- **WaveletBlock**: Multiscale frequency domain feature enhancement
- **WeldSimAM**: Directional parameter-free attention for fine weld seams
- **DySample**: Dynamic lightweight upsampling
- **AHSFPN**: Adaptive Hierarchical Scale Feature Pyramid Network
- **4 Detection Scales**: P2 (stride 4) to P5 (stride 32) for small defect detection


In [2]:
# 1. Environment & Path Resolution (Auto-detects Colab vs Local subfolder)
import os
import sys
import pathlib
import subprocess

# Prevent OpenMP runtime conflict on Windows
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# Auto-install dependencies if missing
try:
    import ultralytics
    import PyWavelets
    import albumentations
except ModuleNotFoundError:
    print("📦 Installing required packages (ultralytics, PyWavelets, opencv-python, albumentations, kagglehub)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics", "PyWavelets", "opencv-python", "albumentations", "kagglehub"])

# Check if running on Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🌐 Environment: Google Colab detected")
    repo_url = "https://github.com/abhisekkundu-DS/Weld-YOLO11-SO"
    repo_name = "Weld-YOLO11-SO"
    target_dir = f"/content/{repo_name}"
    if not os.path.exists(target_dir):
        os.system(f"git clone {repo_url} {target_dir}")
    else:
        print("🔄 Pulling latest code from GitHub...")
        os.system(f"cd {target_dir} && git pull origin main")
    os.chdir(target_dir)
    project_root = pathlib.Path(target_dir).resolve()
else:
    print("💻 Environment: Local Workspace detected")
    cwd = pathlib.Path('.').resolve()
    if (cwd / "models" / "weld_yolo11.yaml").exists():
        project_root = cwd
    elif (cwd.parent / "models" / "weld_yolo11.yaml").exists():
        project_root = cwd.parent
        os.chdir(project_root)
    else:
        project_root = cwd

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 2. Register Weld-YOLO11-SO Custom Modules with Ultralytics
from models.modules import register_custom_modules
register_custom_modules()

import torch
print(f"\n✅ Active Working Directory: {os.getcwd()}")
print(f"   PyTorch CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Device: {torch.cuda.get_device_name(0)}")


📦 Installing required packages (ultralytics, PyWavelets, opencv-python, albumentations, kagglehub)...
🌐 Environment: Google Colab detected
🔄 Pulling latest code from GitHub...
Successfully registered and patched Weld-YOLO11-SO custom modules with Ultralytics framework.

✅ Active Working Directory: /content/Weld-YOLO11-SO
   PyTorch CUDA Available: False


In [2]:
# Verify Weld-YOLO11-SO Model Architecture
from ultralytics import YOLO

model_yaml = project_root / "models" / "weld_yolo11.yaml"
print(f"Loading model architecture from: {model_yaml}")

model = YOLO(str(model_yaml), task="detect")
print("\n✅ Weld-YOLO11-SO model architecture loaded and verified successfully!")


Loading model architecture from: /content/Weld-YOLO11-SO/models/weld_yolo11.yaml
WARNING ⚠️ no model scale passed. Assuming scale='n'.

✅ Weld-YOLO11-SO model architecture loaded and verified successfully!


In [3]:
# Locate or Download Welding Defect Dataset via KaggleHub
from pathlib import Path
import kagglehub

try:
    dataset_path = Path(kagglehub.dataset_download("sukmaadhiwijaya/welding-defect-object-detection"))
    print(f"✅ Kaggle Welding Dataset ready at: {dataset_path}")
except Exception as e:
    print(f"Kaggle download fallback: {e}")
    dataset_path = project_root / "welding_dataset"


Using Colab cache for faster access to the 'welding-defect-object-detection' dataset.
✅ Kaggle Welding Dataset ready at: /kaggle/input/welding-defect-object-detection


In [4]:
# Augmentation Pipeline tailored for Welding Defect Detection
import cv2
import albumentations as A

def create_augmentation_pipeline():
    return A.Compose([
        A.HorizontalFlip(p=0.50),
        A.Rotate(limit=10, border_mode=cv2.BORDER_REFLECT_101, p=0.50),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.50),
        A.Affine(scale=(0.90, 1.10), translate_percent=(-0.05, 0.05), border_mode=cv2.BORDER_REFLECT_101, p=0.50),
        A.GaussianBlur(blur_limit=(3, 3), p=0.10)
    ], bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"], min_visibility=0.20))

print("✅ Albumentations pipeline initialized.")


✅ Albumentations pipeline initialized.


In [5]:
# Dataset Configuration Setup
output_dir = project_root / "Welding_Augmented"
for split in ["train", "valid", "test"]:
    (output_dir / split / "images").mkdir(parents=True, exist_ok=True)
    (output_dir / split / "labels").mkdir(parents=True, exist_ok=True)

data_yaml_content = f"""path: {output_dir.absolute()}
train: train/images
val: valid/images
test: test/images

nc: 3
names:
  0: Bad Weld
  1: Good Weld
  2: Defect
"""
data_yaml = output_dir / "data.yaml"
data_yaml.write_text(data_yaml_content)
print(f"✅ Dataset configuration written to: {data_yaml}")


✅ Dataset configuration written to: /content/Weld-YOLO11-SO/Welding_Augmented/data.yaml


In [6]:
# 🚀 Fast & Optimized Weld-YOLO11-SO Model Training
from ultralytics import YOLO
import os
import torch

train_data_yaml = project_root / "Welding_Augmented" / "data.yaml"
has_images = any((output_dir / "train" / "images").iterdir())
if not has_images:
    print("💡 Local small test mode: Generating benchmark dataset...")
    from train_mnist import prepare_mnist_yolo_dataset
    train_data_yaml = prepare_mnist_yolo_dataset(project_root / "mnist_dataset", num_samples=100)

print(f"🚀 Starting speed-optimized training with config: {train_data_yaml}")
model = YOLO(str(project_root / "models" / "weld_yolo11.yaml"), task="detect")

# SPEED OPTIMIZATION PARAMETERS:
# 1. imgsz=640 (2.5x faster iteration) or 1024 for maximum small-defect precision
# 2. batch=8 (maximizes GPU Tensor Core utilization)
# 3. amp=True (FP16 Automatic Mixed Precision - 2x GPU speedup)
# 4. cache='ram' (caches images in RAM to eliminate disk I/O bottlenecks)
# 5. workers=4 (multi-threaded dataloading on Linux/Colab)
results = model.train(
    data=str(train_data_yaml),
    imgsz=640 if not IN_COLAB else 1024,  # 640 for fast local testing, 1024 in Colab GPU
    epochs=3 if not IN_COLAB else 20,
    batch=8 if torch.cuda.is_available() else 2,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4 if not os.name == 'nt' else 0,
    amp=True,
    cache='ram',
    optimizer='AdamW',
    project="runs/weld_training",
    name="weld_yolo11_so",
    exist_ok=True
)

print("\n✅ Speed-Optimized Training Completed Successfully!")


💡 Local small test mode: Generating 100-sample dataset for model validation...
Successfully registered and patched Weld-YOLO11-SO custom modules with Ultralytics framework.
1. Preparing 100 MNIST digit samples in YOLO format...
🚀 Starting training with data config: /content/Weld-YOLO11-SO/mnist_dataset/data.yaml
WARNING ⚠️ no model scale passed. Assuming scale='n'.
Ultralytics 8.4.132 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Weld-YOLO11-SO/mnist_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, e

: 

In [ ]:
# Evaluate Metrics
metrics = model.val()
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}")
